In [0]:
from pyspark.sql import functions as F

trips = spark.table("urban_mobility.silver.trips_enriched").filter(F.col("trip_status") == "COMPLETED")

hourly_revenue = (
    trips
    .withColumn("hour_start", F.date_trunc("hour", F.col("pickup_datetime")))
    .groupBy("hour_start")
    .agg(
        F.count("trip_id").alias("trip_count"),
        F.round(F.sum("total_amount"), 2).alias("gross_revenue"),
        F.round(F.sum("tip_amount"), 2).alias("tip_revenue"),
        F.round(F.sum("toll_amount"), 2).alias("toll_revenue"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("total_amount"), 2).alias("avg_trip_value"),
        F.round(F.sum("total_amount") / F.sum("distance_km"), 2).alias("revenue_per_km"),
    )
    .orderBy("hour_start")
)

hourly_revenue.write.mode("overwrite").format("delta").saveAsTable("urban_mobility.gold.hourly_revenue")

result = spark.table("urban_mobility.gold.hourly_revenue")
print("gold.hourly_revenue rows:", result.count())
result.orderBy(F.desc("gross_revenue")).show(5)